In [10]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import expr, col

import ConnectionConfig as cc
cc.setupEnvironment()

In [11]:
spark = cc.startLocalCluster("STATION_DIM",4)
spark.getActiveSession()

In [12]:
# load initial station data
df_stations = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stations") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "stationid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()


df_stations.show()


+---------+--------+---------+------------+--------------------+-------+-------+----------+-----------------+--------------------+-------+------+
|stationid|objectid|stationnr|        type|              street| number|zipcode|  district|         gpscoord|      additionalinfo|labelid|cityid|
+---------+--------+---------+------------+--------------------+-------+-------+----------+-----------------+--------------------+-------+------+
|        1|   33202|      026|DUBBELZIJDIG|         Meir (2000)|     84|   2000| ANTWERPEN|(51.2182,4.41241)|                    |   NULL|  NULL|
|        2|   33203|      019| ENKELZIJDIG|          ONTBREKEND|     12|   2000| ANTWERPEN| (51.219,4.40405)|                    |   NULL|  NULL|
|        3|   33204|      020| ENKELZIJDIG|Groenkerkhofstraa...|      2|   2000| ANTWERPEN|(51.2187,4.40066)| thv Nationalestraat|   NULL|  NULL|
|        4|   33205|      035| ENKELZIJDIG|Cockerillkaai (2000)|       |   2000| ANTWERPEN|(51.2104,4.38772)|               

In [13]:
from pyspark.sql.functions import expr

df_station_dim = df_stations.select(
    expr("uuid()").alias("station_sk"),
    col("stationid"),
    col("stationnr"),
    col("street"),
    col("number"),
    col("zipcode"),
    col("district"),
    col("gpscoord")
)

df_station_dim.show()


+--------------------+---------+---------+--------------------+-------+-------+----------+-----------------+
|          station_sk|stationid|stationnr|              street| number|zipcode|  district|         gpscoord|
+--------------------+---------+---------+--------------------+-------+-------+----------+-----------------+
|1f7dbcf3-3232-40d...|        1|      026|         Meir (2000)|     84|   2000| ANTWERPEN|(51.2182,4.41241)|
|35f9339f-15b7-46a...|        2|      019|          ONTBREKEND|     12|   2000| ANTWERPEN| (51.219,4.40405)|
|95f7eeca-f281-440...|        3|      020|Groenkerkhofstraa...|      2|   2000| ANTWERPEN|(51.2187,4.40066)|
|febcfd78-5a6e-4fd...|        4|      035|Cockerillkaai (2000)|       |   2000| ANTWERPEN|(51.2104,4.38772)|
|8328d75d-a4c4-407...|        5|      094|        PALEISSTRAAT|    147|   2018| ANTWERPEN|(51.2047,4.39625)|
|319b9d5c-f567-4f7...|        6|      012|       Singel (2018)|       |   2018| ANTWERPEN|(51.1994,4.39013)|
|3cb919e7-7756-411.

In [14]:
spark.sql("DROP TABLE IF EXISTS stationdim")

df_station_dim.write.format("delta").mode("overwrite").saveAsTable("stationdim")


In [15]:
# export to database
df_station_dim.write \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stationdim") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("batchsize", 1000) \
    .mode("overwrite") \
    .save()


In [16]:
df_verify = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stationdim") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_verify.show()


+--------------------+---------+---------+--------------------+------+-------+---------+-----------------+
|          station_sk|stationid|stationnr|              street|number|zipcode| district|         gpscoord|
+--------------------+---------+---------+--------------------+------+-------+---------+-----------------+
|84b54607-d7b1-46d...|      250|      297|Jan De Voslei (2020)|      |   2020|ANTWERPEN| (51.1907,4.3889)|
|cb7ee1df-74a4-492...|      251|      298|   Valkstraat (2610)|      |   2610|  WILRIJK|(51.1719,4.38248)|
|ede3d861-7f43-485...|      252|      299|Camille Huysmansl...|      |   2020|ANTWERPEN| (51.192,4.39806)|
|bdabf0ee-7178-4b7...|      253|      300|Vogelzanglaan (2020)|      |   2020|ANTWERPEN|(51.1894,4.39731)|
|8c791be5-1f2e-48e...|      254|      301|Luchthavenlei (2100)|     1|   2100|   DEURNE|(51.1888,4.45034)|
|62d17c0f-6613-45d...|      255|      302|Rozenkransplein (...|    11|   2610|  WILRIJK|(51.1791,4.39045)|
|4b6dc95b-26db-4ef...|      256|     

In [23]:
spark.stop()


WHAT GOES DOWN IS FOR TESTING REASONS, RUN IT ONLY FOR IT(ITS ALREADY BEEN TESTED BY RUFINA, SO YOU DONT NEED TO DO IT AGAIN;)


In [17]:
# Load data again to test the initial load
df_stations_test = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stations") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

# Display the data to confirm the load
df_stations_test.show(truncate=False)

+---------+--------+---------+------------+------------------------------+-------+-------+----------+-----------------+-----------------------------+-------+------+
|stationid|objectid|stationnr|type        |street                        |number |zipcode|district  |gpscoord         |additionalinfo               |labelid|cityid|
+---------+--------+---------+------------+------------------------------+-------+-------+----------+-----------------+-----------------------------+-------+------+
|1        |33202   |026      |DUBBELZIJDIG|Meir (2000)                   |84     |2000   |ANTWERPEN |(51.2182,4.41241)|                             |NULL   |NULL  |
|2        |33203   |019      |ENKELZIJDIG |ONTBREKEND                    |12     |2000   |ANTWERPEN |(51.219,4.40405) |                             |NULL   |NULL  |
|3        |33204   |020      |ENKELZIJDIG |Groenkerkhofstraat (2000)     |2      |2000   |ANTWERPEN |(51.2187,4.40066)|thv Nationalestraat          |NULL   |NULL  |
|4        

In [18]:
# Check data in the Delta table
spark.sql("SELECT * FROM stationdim").show(truncate=False)

# Check the table schema
spark.sql("DESCRIBE TABLE stationdim").show(truncate=False)


+------------------------------------+---------+---------+-----------------------------+------+-------+---------+-----------------+
|station_sk                          |stationid|stationnr|street                       |number|zipcode|district |gpscoord         |
+------------------------------------+---------+---------+-----------------------------+------+-------+---------+-----------------+
|84b54607-d7b1-46dc-9997-3738f3605332|250      |297      |Jan De Voslei (2020)         |      |2020   |ANTWERPEN|(51.1907,4.3889) |
|cb7ee1df-74a4-4921-8803-f881ed961003|251      |298      |Valkstraat (2610)            |      |2610   |WILRIJK  |(51.1719,4.38248)|
|ede3d861-7f43-485d-a94e-9e9af960bcc6|252      |299      |Camille Huysmanslaan (2020)  |      |2020   |ANTWERPEN|(51.192,4.39806) |
|bdabf0ee-7178-4b7d-9a54-c845385af7fa|253      |300      |Vogelzanglaan (2020)         |      |2020   |ANTWERPEN|(51.1894,4.39731)|
|8c791be5-1f2e-48e2-852b-dc49e40b5255|254      |301      |Luchthavenlei (210

In [19]:
# load data from PostgreSQL to verify export
df_stationdim_pg = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stationdim") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_stationdim_pg.show(truncate=False)


+------------------------------------+---------+---------+-----------------------------+------+-------+---------+-----------------+
|station_sk                          |stationid|stationnr|street                       |number|zipcode|district |gpscoord         |
+------------------------------------+---------+---------+-----------------------------+------+-------+---------+-----------------+
|84b54607-d7b1-46dc-9997-3738f3605332|250      |297      |Jan De Voslei (2020)         |      |2020   |ANTWERPEN|(51.1907,4.3889) |
|cb7ee1df-74a4-4921-8803-f881ed961003|251      |298      |Valkstraat (2610)            |      |2610   |WILRIJK  |(51.1719,4.38248)|
|ede3d861-7f43-485d-a94e-9e9af960bcc6|252      |299      |Camille Huysmanslaan (2020)  |      |2020   |ANTWERPEN|(51.192,4.39806) |
|bdabf0ee-7178-4b7d-9a54-c845385af7fa|253      |300      |Vogelzanglaan (2020)         |      |2020   |ANTWERPEN|(51.1894,4.39731)|
|8c791be5-1f2e-48e2-852b-dc49e40b5255|254      |301      |Luchthavenlei (210

In [20]:
#after insertion the following line in the sql console:
# UPDATE stations SET street = 'New Street Name' WHERE stationid = 1;





# load the updated stations data
df_stations_updated = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stations") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_stations_updated.show()

# Transform and prepare the data again
df_station_dim_updated = df_stations_updated.select(
    expr("uuid()").alias("station_sk"),
    col("stationid"),
    col("stationnr"),
    col("street"),
    col("number"),
    col("zipcode"),
    col("district"),
    col("gpscoord")
)

df_station_dim_updated.show()


+---------+--------+---------+------------+--------------------+-------+-------+----------+-----------------+--------------------+-------+------+
|stationid|objectid|stationnr|        type|              street| number|zipcode|  district|         gpscoord|      additionalinfo|labelid|cityid|
+---------+--------+---------+------------+--------------------+-------+-------+----------+-----------------+--------------------+-------+------+
|        2|   33203|      019| ENKELZIJDIG|          ONTBREKEND|     12|   2000| ANTWERPEN| (51.219,4.40405)|                    |   NULL|  NULL|
|        3|   33204|      020| ENKELZIJDIG|Groenkerkhofstraa...|      2|   2000| ANTWERPEN|(51.2187,4.40066)| thv Nationalestraat|   NULL|  NULL|
|        4|   33205|      035| ENKELZIJDIG|Cockerillkaai (2000)|       |   2000| ANTWERPEN|(51.2104,4.38772)|                    |   NULL|  NULL|
|        5|   33206|      094| ENKELZIJDIG|        PALEISSTRAAT|    147|   2018| ANTWERPEN|(51.2047,4.39625)|               

In [22]:
# Check the Delta table again to confirm no changes
spark.sql("SELECT * FROM stationdim WHERE stationid=1").show(truncate=False)


+------------------------------------+---------+---------+-----------+------+-------+---------+-----------------+
|station_sk                          |stationid|stationnr|street     |number|zipcode|district |gpscoord         |
+------------------------------------+---------+---------+-----------+------+-------+---------+-----------------+
|1f7dbcf3-3232-40da-87a2-44313faf82d4|1        |026      |Meir (2000)|84    |2000   |ANTWERPEN|(51.2182,4.41241)|
+------------------------------------+---------+---------+-----------+------+-------+---------+-----------------+



It works well, after running an update command in the console:UPDATE stations SET street = 'New Street Name' WHERE stationid = 1;

The Delta table still shows the original data, even if the source PostgreSQL table has changed.
This confirms that Type 0 is respected.